In [ ]:
!pip install -q kaggle
import tensorflow as tf
print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))


In [ ]:
# --- Option A: Download from Kaggle (matches the original lab report) ---
from google.colab import files
print("Upload your kaggle.json API token:")
uploaded = files.upload()

import os
os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

!kaggle datasets download -d valentynsichkar/cifar10-preprocessed -p /content/cifar10 --unzip
# If the above dataset slug doesn't work, search Kaggle for "CIFAR-10 python batches"
# and substitute the correct dataset slug.


In [ ]:
# --- Option B: Load CIFAR-10 directly via Keras (simpler, no Kaggle needed) ---
# Comment this cell out if you used Option A above.

import numpy as np
from tensorflow.keras.datasets import cifar10

(x_train_full, y_train_full), (x_test_full, y_test_full) = cifar10.load_data()

class_names = ['Airplane', 'Automobile', 'Bird', 'Cat', 'Deer',
               'Dog', 'Frog', 'Horse', 'Ship', 'Truck']

print("Full training data shape:", x_train_full.shape, y_train_full.shape)
print("Full testing data shape :", x_test_full.shape, y_test_full.shape)


In [ ]:
# --- If you used Option A (Kaggle pickled batches) instead, use this loader ---

import pickle, glob, os
import numpy as np

def unpickle(file):
    with open(file, 'rb') as fo:
        return pickle.load(fo, encoding='bytes')

matches = glob.glob('/content/cifar10/**/data_batch_1', recursive=True)
if matches:
    base_dir = os.path.dirname(matches[0])

    x_train_list, y_train_list = [], []
    for i in range(1, 6):
        batch = unpickle(os.path.join(base_dir, f'data_batch_{i}'))
        x_train_list.append(batch[b'data'])
        y_train_list.extend(batch[b'labels'])

    x_train_full = np.concatenate(x_train_list).reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
    y_train_full = np.array(y_train_list).reshape(-1, 1)

    test_batch = unpickle(os.path.join(base_dir, 'test_batch'))
    x_test_full = test_batch[b'data'].reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
    y_test_full = np.array(test_batch[b'labels']).reshape(-1, 1)

    class_names = ['Airplane', 'Automobile', 'Bird', 'Cat', 'Deer',
                   'Dog', 'Frog', 'Horse', 'Ship', 'Truck']

    print("Loaded from Kaggle batches.")
else:
    print("No Kaggle batches found — continuing with the Keras-loaded arrays from Option B.")


In [ ]:
# Take a stratified random subset to keep runtime manageable
# (20,000 train / 4,000 test out of the full 50,000 / 10,000)

np.random.seed(42)
train_idx = np.random.choice(len(x_train_full), 20000, replace=False)
test_idx = np.random.choice(len(x_test_full), 4000, replace=False)

x_train, y_train = x_train_full[train_idx], y_train_full[train_idx]
x_test, y_test = x_test_full[test_idx], y_test_full[test_idx]

# Normalize pixel values to [0, 1]
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

print("Training data shape:", x_train.shape, y_train.shape)
print("Testing data shape :", x_test.shape, y_test.shape)


In [ ]:
# Display ten sample images

import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(x_train[i])
    ax.set_title(class_names[int(y_train[i])], fontweight='bold')
    ax.axis('off')
plt.suptitle("Sample CIFAR-10 Images", fontsize=14)
plt.tight_layout()
plt.savefig('sample_images.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Input, Resizing
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical

IMG_SIZE = 96  # MobileNetV2 expects inputs of at least 32x32; 96x96 preserves more detail than 32x32

# One-hot encode labels
y_train_cat = to_categorical(y_train, num_classes=10)
y_test_cat = to_categorical(y_test, num_classes=10)

# Load pretrained ImageNet weights and freeze the convolutional base
base_model = MobileNetV2(input_shape=(IMG_SIZE, IMG_SIZE, 3),
                          include_top=False,
                          weights='imagenet')
base_model.trainable = False

# Build the full model: resize -> MobileNetV2 base -> GAP -> Dense(ReLU) -> Dense(Softmax)
inputs = Input(shape=(32, 32, 3))
x = Resizing(IMG_SIZE, IMG_SIZE)(inputs)
x = base_model(x, training=False)
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
outputs = Dense(10, activation='softmax')(x)

model = Model(inputs, outputs)
model.compile(optimizer=Adam(learning_rate=0.001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.summary()


In [ ]:
import time

start_time = time.time()

history = model.fit(
    x_train, y_train_cat,
    validation_data=(x_test, y_test_cat),
    batch_size=64,
    epochs=8
)

stage1_time = (time.time() - start_time) / 60
print(f"Stage 1 training time: {stage1_time:.2f} minutes")


In [ ]:
# Unfreeze the last convolutional block (last 20 layers) and continue training
# at a much smaller learning rate

base_model.trainable = True
fine_tune_at = len(base_model.layers) - 20

for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(optimizer=Adam(learning_rate=0.0001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

start_time = time.time()

history_ft = model.fit(
    x_train, y_train_cat,
    validation_data=(x_test, y_test_cat),
    batch_size=64,
    epochs=5
)

finetune_time = (time.time() - start_time) / 60
total_time = stage1_time + finetune_time
print(f"Fine-tuning time: {finetune_time:.2f} minutes")
print(f"Total training time: {total_time:.2f} minutes")


In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)
import numpy as np

y_pred_probs = model.predict(x_test, batch_size=64)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = y_test.flatten()

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average='weighted')
recall = recall_score(y_true, y_pred, average='weighted')
f1 = f1_score(y_true, y_pred, average='weighted')
cm = confusion_matrix(y_true, y_pred)
report = classification_report(y_true, y_pred, target_names=class_names)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")
print()
print("Classification Report:")
print(report)


In [ ]:
# Plot: Training vs Validation Accuracy (across both frozen-base and fine-tuning phases)

acc = history.history['accuracy'] + history_ft.history['accuracy']
val_acc = history.history['val_accuracy'] + history_ft.history['val_accuracy']
epochs_range = range(1, len(acc) + 1)
fine_tune_epoch = len(history.history['accuracy'])

plt.figure(figsize=(8, 6))
plt.plot(epochs_range, acc, label='Training Accuracy', linewidth=2)
plt.plot(epochs_range, val_acc, label='Validation Accuracy', linewidth=2)
plt.axvline(x=fine_tune_epoch, color='gray', linestyle='--', label='Fine-tuning starts')
plt.title('Training vs Validation Accuracy', fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(alpha=0.3)
plt.savefig('accuracy_plot.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Plot: Training vs Validation Loss

loss = history.history['loss'] + history_ft.history['loss']
val_loss = history.history['val_loss'] + history_ft.history['val_loss']

plt.figure(figsize=(8, 6))
plt.plot(epochs_range, loss, label='Training Loss', linewidth=2)
plt.plot(epochs_range, val_loss, label='Validation Loss', linewidth=2)
plt.axvline(x=fine_tune_epoch, color='gray', linestyle='--', label='Fine-tuning starts')
plt.title('Training vs Validation Loss', fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(alpha=0.3)
plt.savefig('loss_plot.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Plot: Confusion Matrix

import seaborn as sns

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names,
            cbar_kws={'label': 'Count'})
plt.title('Confusion Matrix — MobileNetV2 (Transfer Learning)', fontweight='bold')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Plot: Misclassified Images (Optional)

misclassified_idx = np.where(y_pred != y_true)[0]
sample_idx = np.random.choice(misclassified_idx, size=min(10, len(misclassified_idx)), replace=False)

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for ax, idx in zip(axes.flat, sample_idx):
    ax.imshow(x_test[idx])
    ax.set_title(f"T:{class_names[y_true[idx]]}\nP:{class_names[y_pred[idx]]}", fontsize=10, fontweight='bold')
    ax.axis('off')
plt.suptitle("Misclassified Images", fontsize=14)
plt.tight_layout()
plt.savefig('misclassified.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
frozen_train_acc = history.history['accuracy'][-1]
frozen_val_acc = history.history['val_accuracy'][-1]
frozen_val_loss = history.history['val_loss'][-1]

ft_train_acc = history_ft.history['accuracy'][-1]
ft_val_acc = history_ft.history['val_accuracy'][-1]
ft_val_loss = history_ft.history['val_loss'][-1]

print("Metric                Frozen Base    Fine-Tuned")
print(f"Training Accuracy     {frozen_train_acc:.4f}         {ft_train_acc:.4f}")
print(f"Validation Accuracy   {frozen_val_acc:.4f}         {ft_val_acc:.4f}")
print(f"Validation Loss       {frozen_val_loss:.4f}         {ft_val_loss:.4f}")
print(f"Training Time (min)   {stage1_time:.2f}           {finetune_time:.2f}")


In [ ]:
import pandas as pd

summary = pd.DataFrame({
    "Metric": ["Training Accuracy (final)", "Testing Accuracy", "Precision",
               "Recall", "F1-score", "Total Training Time (min)", "Total Parameters"],
    "Value": [f"{ft_train_acc:.4f}", f"{accuracy:.4f}", f"{precision:.4f}",
              f"{recall:.4f}", f"{f1:.4f}", f"{total_time:.2f}",
              f"{model.count_params():,}"]
})
summary
